# TFT - Weekly Revenue Forecast (8-week horizon)

Ноутбук повторяет структуру статьи и минимально адаптирует её под weekly `City`-уровень проекта.

## 1. Загрузка и подготовка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from typing import List

from darts import TimeSeries
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler, StaticCovariatesTransformer
from darts.utils.likelihood_models import QuantileRegression
from darts.explainability import TFTExplainer

import torch
import torchmetrics

from sklearn.preprocessing import MinMaxScaler, StandardScaler, OrdinalEncoder
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks.lr_monitor import LearningRateMonitor

## 2. Формирование датафреймов target и covariates

In [ ]:
df = pd.read_csv("data/sales.csv", sep=";")
df["date"] = pd.to_datetime(df["Week"])
df = (
    df.groupby(["City", "date"], as_index=False)["revenue"]
    .sum()
    .sort_values(["City", "date"])
    .reset_index(drop=True)
)

city_frames = []
for city, city_frame in df.groupby("City", sort=True):
    full_weeks = pd.date_range(city_frame["date"].min(), city_frame["date"].max(), freq="W-MON")
    city_grid = pd.DataFrame({"date": full_weeks})
    city_grid["City"] = city
    city_grid = city_grid.merge(city_frame, on=["City", "date"], how="left")
    city_grid["is_missing"] = city_grid["revenue"].isna().astype(int)
    city_grid["revenue"] = city_grid["revenue"].fillna(0.0)
    city_frames.append(city_grid)

df = pd.concat(city_frames, ignore_index=True).sort_values(["City", "date"]).reset_index(drop=True)
print(f"Series: {df['City'].nunique()}, Date range: {df['date'].min()} - {df['date'].max()}")
print(f"Total rows: {len(df)}")

In [ ]:
iso_calendar = df["date"].dt.isocalendar()
df["week_of_year"] = iso_calendar.week.astype(int)
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["year"] = df["date"].dt.year
df["is_holiday_week"] = 0

shifted = df.groupby("City")["revenue"].shift(1)
df["lag_1"] = shifted
df["lag_2"] = df.groupby("City")["revenue"].shift(2)
df["lag_4"] = df.groupby("City")["revenue"].shift(4)
df["lag_8"] = df.groupby("City")["revenue"].shift(8)
df["rolling_4"] = shifted.groupby(df["City"]).rolling(4).mean().reset_index(level=0, drop=True)
df["rolling_8"] = shifted.groupby(df["City"]).rolling(8).mean().reset_index(level=0, drop=True)
df["rolling_12"] = shifted.groupby(df["City"]).rolling(12).mean().reset_index(level=0, drop=True)

df_target = df[["date", "City", "revenue"]].copy()
df_past_covs = df[
    ["date", "City", "lag_1", "lag_2", "lag_4", "lag_8", "rolling_4", "rolling_8", "rolling_12"]
].copy()
df_future_covs = df[["date", "City", "week_of_year", "month", "quarter", "year", "is_holiday_week"]].copy()

display(df_target.head())
display(df_past_covs.head())
display(df_future_covs.head())

## 3. Формирование TimeSeries

In [ ]:
ts_target_list = TimeSeries.from_group_dataframe(
    df=df_target.assign(City_static=df_target["City"]),
    group_cols="City",
    time_col="date",
    value_cols="revenue",
    static_cols=["City_static"],
    fill_missing_dates=True,
    freq="W-MON",
    fillna_value=0.0,
)

ts_past_covs_list = TimeSeries.from_group_dataframe(
    df=df_past_covs.fillna(0.0),
    group_cols="City",
    time_col="date",
    value_cols=["lag_1", "lag_2", "lag_4", "lag_8", "rolling_4", "rolling_8", "rolling_12"],
    fill_missing_dates=True,
    freq="W-MON",
    fillna_value=0.0,
)

future_rows = []
future_dates = pd.date_range(df["date"].max() + pd.Timedelta(weeks=1), periods=8, freq="W-MON")
for city in sorted(df_future_covs["City"].unique()):
    for future_date in future_dates:
        future_rows.append({"date": future_date, "City": city})

future_extension = pd.DataFrame(future_rows)
future_extension["week_of_year"] = future_extension["date"].dt.isocalendar().week.astype(int)
future_extension["month"] = future_extension["date"].dt.month
future_extension["quarter"] = future_extension["date"].dt.quarter
future_extension["year"] = future_extension["date"].dt.year
future_extension["is_holiday_week"] = 0

df_future_covs_extended = pd.concat([df_future_covs, future_extension], ignore_index=True)
ts_future_covs_list = TimeSeries.from_group_dataframe(
    df=df_future_covs_extended,
    group_cols="City",
    time_col="date",
    value_cols=["week_of_year", "month", "quarter", "year", "is_holiday_week"],
    fill_missing_dates=True,
    freq="W-MON",
    fillna_value=0.0,
)

print(f"Total series: {len(ts_target_list)}")

## 4. Нормализация

In [ ]:
target_series_scaler = Scaler(scaler=StandardScaler(), global_fit=False)
target_static_scaler = StaticCovariatesTransformer()
past_covs_scaler = Scaler(scaler=StandardScaler(), global_fit=False)
future_covs_scaler = Scaler(scaler=MinMaxScaler(feature_range=(0, 1)), global_fit=True)

ts_target_list = target_series_scaler.fit_transform(ts_target_list)
ts_target_list = target_static_scaler.fit_transform(ts_target_list)
ts_past_covs_list = past_covs_scaler.fit_transform(ts_past_covs_list)
ts_future_covs_list = future_covs_scaler.fit_transform(ts_future_covs_list)

## 5. Разделение на train / val / test

In [ ]:
input_len = 60
output_len = 8
window_len = input_len + output_len

max_dt = df["date"].max()
test_min_dt = max_dt - pd.Timedelta(weeks=output_len - 1)
test_context_min_dt = test_min_dt - pd.Timedelta(weeks=input_len)
val_max_dt = test_min_dt - pd.Timedelta(weeks=1)
val_min_dt = val_max_dt - pd.Timedelta(weeks=window_len - 1)
train_min_dt = df["date"].min()
train_max_dt = val_min_dt - pd.Timedelta(weeks=1)

def series_splitter(series_list: List[TimeSeries]):
    train: List[TimeSeries] = []
    val: List[TimeSeries] = []
    test: List[TimeSeries] = []

    for series in tqdm(series_list):
        train_series = series[train_min_dt:train_max_dt]
        val_series = series[val_min_dt:val_max_dt]
        test_series = series[test_context_min_dt:max_dt]

        if len(train_series) >= window_len:
            train.append(train_series)
        if len(val_series) >= window_len:
            val.append(val_series)
        if len(test_series) >= window_len:
            test.append(test_series)

    return train, val, test

ts_target_train, ts_target_val, ts_target_test = series_splitter(ts_target_list)
ts_past_covs_train, ts_past_covs_val, ts_past_covs_test = series_splitter(ts_past_covs_list)
ts_future_covs_train, ts_future_covs_val, ts_future_covs_test = series_splitter(ts_future_covs_list)

## 6. Создание и обучение TFT

In [ ]:
# заполним на следующем шаге

## 7. Выполнение прогнозов

In [ ]:
# заполним на следующем шаге

## 8. Оценка качества и визуализация

In [ ]:
# заполним на следующем шаге

## 9. Интерпретация результатов

In [ ]:
# заполним на следующем шаге